# Baseline corpora build (MedRAG and PaperQA2, article-only)

Rebuilds both benchmark baselines' corpora from a dated corpus build produced by
`srv/nbs/1_download_ds_papers.ipynb`, so that every benchmark arm (T21, MedRAG,
PaperQA2) answers from the identical document set.

Selection rule, identical to the embedding notebook (`2_embed_and_index_ds_papers.ipynb`):
papers with at least 4 of the 6 target sections, at least one retrieval section, AND
`type == "article"` (original research only), read from the build's `master_table.csv`.
For the 2026-09-02 build that is 1,594 papers (158 INCLUDE), matching the live
collection DS_INCLUDE_2026_09_02_DEEPINFRA_2048 (65,677 vectors / 1,594 documents)
exactly. Using the same rule everywhere is the point: if the T21
index and the baselines disagree about the corpus, the comparison stops being
controlled.

Outputs (all date-stamped, previous corpora untouched):

- MedRAG: `analysis/benchmarks/MedRAG/corpus/t21_<date>/chunk/*.jsonl`, flat
  approximately 1,000 character chunks with no section labels, MedRAG's own convention,
  plus its MedCPT index (built in this notebook, roughly 35 minutes on CPU).
- PaperQA2: `analysis/benchmarks/paperqa_corpus_<date>/*.txt`, one text file per paper
  with a citation header and section headings, PaperQA2 chunks these itself.
- A manifest CSV for each, so the exact corpus behind any benchmark number is on record.

Environment: the `medrag` kernel (the MedCPT index build needs it). The corpus text
generation itself is plain Python and environment-neutral.


In [1]:
# ---- Cell 1: configuration ----
import csv, json, re, shutil, sys, time
from pathlib import Path

import pandas as pd

NB_DIR   = Path.cwd()
ANALYSIS = NB_DIR.parent if NB_DIR.name == "nbs" else NB_DIR
BENCH    = ANALYSIS / "benchmarks"
SRV      = ANALYSIS.parent / "srv"

RUN_DATE = "2026-09-02"          # which corpus build to mirror

RUN_DIR  = SRV / "output" / f"ds_include_pipeline_{RUN_DATE}"
JSON_DIR = RUN_DIR / "json_cache"

DATE_TAG            = RUN_DATE.replace("-", "_")
MEDRAG_CORPUS_NAME  = f"T21_{DATE_TAG}_ARTICLES"              # e.g. T21_2026_09_02_ARTICLES
MEDRAG_CHUNK_DIR    = BENCH / "MedRAG" / "corpus" / MEDRAG_CORPUS_NAME.lower() / "chunk"
PQA_CORPUS_DIR      = BENCH / f"paperqa_corpus_{RUN_DATE}_articles"

SECTION_ORDER = ["Abstract", "Introduction", "Methods", "Results", "Discussion", "Conclusion"]

assert (RUN_DIR / "master_table.csv").exists(), f"no completed build at {RUN_DIR}"
sys.path.insert(0, str(BENCH))
print("source build      :", RUN_DIR.name)
print("MedRAG corpus     :", MEDRAG_CORPUS_NAME, "->", MEDRAG_CHUNK_DIR)
print("PaperQA2 corpus   :", PQA_CORPUS_DIR.name)


source build      : ds_include_pipeline_2026-09-02
MedRAG corpus     : T21_2026_09_02_ARTICLES -> /path/to/project/T21_research_assistant_v7/analysis/benchmarks/MedRAG/corpus/t21_2026_09_02_articles/chunk
PaperQA2 corpus   : paperqa_corpus_2026-09-02_articles


## Cell 2: select the papers (article-only, >=4 sections)

Same selection the T21 embedding notebook uses, from the same master table, so all
three arms are guaranteed to share one corpus definition rather than three separately
maintained ones.


In [2]:
# ---- Cell 2: selection ----
def present(v):
    return v is not None and str(v).strip() not in ("", "N/A", "NA", "n/a", "na")

m = pd.read_csv(RUN_DIR / "master_table.csv")
keep = m[(m["n_sections"] >= 4) & (m["n_retrieval"] > 0)
         & (m["type"] == "article")].copy()   # identical to notebook 2 of srv/nbs
keep["pmid"] = keep["pmid"].astype(str)
print(f"papers in build       : {len(m)}")
print(f"selected (article, >=4 sections): {len(keep)}")
assert len(keep) == 1594, f"expected 1,594 selected papers, got {len(keep)}"
assert int(keep["is_include"].sum()) == 158
print(f"INCLUDE-tagged        : {int(keep['is_include'].sum())}")

papers = []
for _, r in keep.iterrows():
    f = JSON_DIR / f"publication_metadata_PMID_{r['pmid']}.json"
    if not f.exists():
        continue
    d = json.loads(f.read_text())
    papers.append(d)
print(f"JSONs loaded          : {len(papers)}")
assert len(papers) == len(keep), "selected papers missing from json_cache"


papers in build       : 1837
selected (article, >=4 sections): 1594
INCLUDE-tagged        : 158
JSONs loaded          : 1594


## Cell 3: MedRAG corpus

MedRAG's own snippet convention: flat text, sentence-aware splits at about 1,000
characters (their Textbooks corpus measures a mean of 810), no section labels, title
set to the PMCID so retrieved snippets stay traceable to a paper. Sections are
concatenated in reading order, so MedRAG also sees Abstract and Introduction, which a
generic RAG would use and the T21 retrieval filter deliberately does not.


In [3]:
# ---- Cell 3: MedRAG jsonl corpus ----
from build_t21_medrag_corpus import split_text     # reuse the tested splitter

if MEDRAG_CHUNK_DIR.exists():
    shutil.rmtree(MEDRAG_CHUNK_DIR.parent)
MEDRAG_CHUNK_DIR.mkdir(parents=True)

PAPERS_PER_FILE = 200
manifest, buf, part, n_chunks = [], [], 0, 0
for d in papers:
    pmid = str(d.get("pmid"))
    pmcid = str(d.get("pmcid") or "").strip() or pmid
    secs = d.get("sections") or {}
    body = "\n\n".join(str(secs[s]).strip() for s in SECTION_ORDER if present(secs.get(s)))
    if not body:
        continue
    pieces = split_text(body)
    for i, c in enumerate(pieces):
        buf.append({"id": f"{pmcid}_{i}", "title": pmcid, "content": c})
    n_chunks += len(pieces)
    manifest.append({"pmid": pmid, "pmcid": pmcid, "chunks": len(pieces), "chars": len(body)})
    if len(manifest) % PAPERS_PER_FILE == 0:
        (MEDRAG_CHUNK_DIR / f"t21_{part:03d}.jsonl").write_text(
            "\n".join(json.dumps(r, ensure_ascii=False) for r in buf) + "\n")
        buf, part = [], part + 1
if buf:
    (MEDRAG_CHUNK_DIR / f"t21_{part:03d}.jsonl").write_text(
        "\n".join(json.dumps(r, ensure_ascii=False) for r in buf) + "\n")

with open(BENCH / f"t21_medrag_manifest_{RUN_DATE}_articles.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["pmid", "pmcid", "chunks", "chars"])
    w.writeheader(); w.writerows(manifest)
print(f"MedRAG corpus: {len(manifest)} papers, {n_chunks:,} chunks, "
      f"{len(list(MEDRAG_CHUNK_DIR.glob('*.jsonl')))} jsonl files")


MedRAG corpus: 1594 papers, 60,391 chunks, 8 jsonl files


## Cell 4: PaperQA2 corpus

One plain-text file per paper with a citation header (title, journal, year, PMCID,
PMID, DOI) and the section headings retained; PaperQA2 does its own chunking at 5,000
characters. Text files rather than the archived PDFs, for the reasons recorded in
notebook 18: complete coverage, identical text to what T21 embeds, and deterministic
parsing.


In [4]:
# ---- Cell 4: PaperQA2 text corpus ----
if PQA_CORPUS_DIR.exists():
    shutil.rmtree(PQA_CORPUS_DIR)
PQA_CORPUS_DIR.mkdir(parents=True)

rows = []
for d in papers:
    pmid = str(d.get("pmid"))
    pmcid = str(d.get("pmcid") or "").strip() or pmid
    secs = d.get("sections") or {}
    body_parts = [f"## {s}\n{str(secs[s]).strip()}"
                  for s in SECTION_ORDER if present(secs.get(s))]
    if not body_parts:
        continue
    header = (f"{str(d.get('title') or '').strip()}\n"
              f"{str(d.get('source') or '').strip()} ({d.get('year') or ''})\n"
              f"PMCID: {pmcid}  PMID: {pmid}\n"
              f"DOI: {d.get('doi') or ''}\n")
    text = header + "\n" + "\n\n".join(body_parts) + "\n"
    (PQA_CORPUS_DIR / f"{pmcid}.txt").write_text(text, encoding="utf-8")
    rows.append({"pmid": pmid, "pmcid": pmcid, "chars": len(text),
                 "sections": len(body_parts)})

with open(BENCH / f"t21_paperqa_manifest_{RUN_DATE}_articles.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["pmid", "pmcid", "chars", "sections"])
    w.writeheader(); w.writerows(rows)
total = sum(r["chars"] for r in rows)
print(f"PaperQA2 corpus: {len(rows)} papers, {total/1e6:.1f} M chars "
      f"(~{total/4/1e6:.1f} M tokens to embed)")


PaperQA2 corpus: 1594 papers, 53.8 M chars (~13.5 M tokens to embed)


## Cell 5: build the MedRAG index

Constructing the retriever triggers the MedCPT embedding of every chunk and the FAISS
index build. Roughly 35 minutes on CPU for a corpus this size; shard embeddings are
cached, so an interrupted build resumes. A smoke query at the end proves retrieval and
generation work against the new corpus before any benchmark uses it.

The PaperQA2 index is NOT built here: it belongs to the paperqa environment and is
built (and pickled) by notebook 18's cell 3 when pointed at this corpus.


In [ ]:
# ---- Cell 5: MedRAG index + smoke query ----
from medrag_adapter import build_medrag, answer_freetext

t0 = time.time()
mr = build_medrag(corpus=MEDRAG_CORPUS_NAME, retriever="MedCPT", provider="ollama_remote")
print(f"index ready in {(time.time()-t0)/60:.1f} min")

txt, snips = answer_freetext(mr, "What is the prevalence of congenital heart disease in Down syndrome?", k=8)
print("snippets:", len(snips), "| papers:", sorted({s.get("title") for s in snips})[:5])
print("ANSWER:", txt[:300])
assert len(snips) > 0 and len(txt) > 50, "smoke query failed"


/home/user/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[In progress] Embedding the t21_2026_09_02_articles corpus with the ncbi/MedCPT-Article-Encoder retriever...


/home/user/anaconda3/envs/medrag/lib/python3.10/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/user/anaconda3/envs/medrag/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'cached_download' (from 'huggingface_hub.file_download') is deprecated and will be removed from version '0.26'. Use `hf_hub_download` instead.
  warnings.warn(warning_message, FutureWarning)


No sentence-transformers model found with name /home/user/.cache/torch/sentence_transformers/ncbi_MedCPT-Article-Encoder. Creating a new one with CLS pooling.


100%|██████████| 8/8 [34:53<00:00, 261.66s/it]


...

100%|██████████| 8/8 [00:00<00:00, 30.75it/s]


[Finished] Corpus indexing finished!


/home/user/anaconda3/envs/medrag/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'cached_download' (from 'huggingface_hub.file_download') is deprecated and will be removed from version '0.26'. Use `hf_hub_download` instead.
  warnings.warn(warning_message, FutureWarning)


No sentence-transformers model found with name /home/user/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.
index ready in 35.0 min
snippets: 8 | papers: ['PMC10201969', 'PMC10496550', 'PMC11192809', 'PMC3848635', 'PMC5370349']
ANSWER: The prevalence of congenital heart disease (CHD) in individuals with Down syndrome varies across studies, but the available data indicate that roughly one‑third to one‑half of those with Down syndrome have CHD. - In a retrospective Egyptian cohort of 1,720 children with Down syndrome, the overall pr


: 

---
## Pointing the benchmark notebooks at these corpora

Notebook 17 (MedRAG benchmark), cell 3: change the corpus argument to

    medrag = build_medrag(corpus="T21_2026_09_02_ARTICLES", retriever="MedCPT", provider="ollama_remote")

or, generically, `corpus=MEDRAG_CORPUS_NAME` for whatever RUN_DATE this notebook used.
Unknown corpus names are auto-registered by the adapter, so no MedRAG source edit is
needed.

Notebook 18 (PaperQA2 benchmark), cell 1: set

    CORPUS_DATE = "2026-09-02_articles"

and delete `paperqa_docs.pkl` in its output directory so the index is rebuilt over the
new corpus (indexes are corpus-specific and must not be reused across corpora).

Record in the benchmark write-up which dated corpus every arm used; the manifests
written here are the citable definition of each.
